# Save the Human (in the loop): when does human oversight of AI actually reduce risk? — Results

This notebook shows the results of the study set out in the project outline. It computes them itself, section by section, from the public FiFAR dataset. Starting from the dataset's alerts file and the framework's Classifier h, it regenerates the simulated reviewers, draws the teams, runs the three policies, re-draws the cases to give every difference a range, and applies the rules that decide the hypothesis. The outputs below are those of the run on free Google Colab that produced this copy. The notebook follows the project outline's sections: Problem, Objective, Hypothesis, Proposed criteria, Method, Risk measures, Simple governance effectiveness monitoring, Integrity of the result, Limitations and Risks. Where the study departs from the project outline, the departure is explained where it first matters. Technical records (run time and memory of this run, input checksums, and how each item traces to the study's design) are at the end and are not needed to read the results.

**Data.** Financial Fraud Alert Review Dataset (FiFAR), version 1 (Alves et al., 2025), figshare, digital object identifier 10.6084/m9.figshare.28351172 (https://doi.org/10.6084/m9.figshare.28351172), licensed under CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/). The dataset is described in Alves et al., "A benchmarking framework and dataset for learning to defer in human-AI decision-making", *Scientific Data* 12, 506 (2025), https://doi.org/10.1038/s41597-025-04664-y. The notebook downloads the dataset and uses it unmodified; the repository does not redistribute it. The dataset's `Base.csv` derives from the Bank Account Fraud dataset, which is credited in the README, section "Data: FiFAR" ([https://github.com/zsipka/SaveHITL-pub#data-fifar](https://github.com/zsipka/SaveHITL-pub#data-fifar)); this notebook downloads that file and checks its checksum, but computes nothing from it.

The tables and graphs this notebook produces are computed from FiFAR (Alves et al., figshare, 2025, https://doi.org/10.6084/m9.figshare.28351172, CC BY 4.0, https://creativecommons.org/licenses/by/4.0/). They are derived from FiFAR by this project, not part of it, and FiFAR's creators have not reviewed or endorsed them.

**Files in `results/`: derived from FiFAR.** This project made the three files in `results/` from its runs on FiFAR (https://doi.org/10.6084/m9.figshare.28351172, CC BY 4.0, https://creativecommons.org/licenses/by/4.0/). They are derived from FiFAR, not part of it.

- `router_selected_settings.csv`: the settings chosen for each of the 250 router models, with the weighted log-loss of the selected trial (column `selected_weighted_log_loss`), which this project computed on FiFAR.
- `router_model_checksums.csv`: the checksums of the 250 router models this project trained on FiFAR.
- `input_checksums.csv`: the paths, sizes and checksums of the input files, 132 from FiFAR and 10 from OpenL2D (Apache License 2.0, see `NOTICE`).

They hold no FiFAR case row and no data value from FiFAR: no feature, label, score, reviewer decision or reviewer parameter. To show what each row refers to, they reuse FiFAR's own reviewer labels (for example `standard#13`) and FiFAR's file paths and file sizes. FiFAR's creators have not reviewed or endorsed these files.

## Running the notebook

**What the run does.** Open the notebook in Google Colab from the repository, or run it on your own machine with the repository's environment file; Colab is the tested route. Run the cells in order. The run:
1. sets up the environment: Python 3.7.9 and pinned library versions;
2. clones the OpenL2D framework at its pinned commit and applies the project's four patches, each of which states its change;
3. downloads the FiFAR files from figshare, and stops if any file's checksum differs from the list in `results/`;
4. computes every section in turn.

It does not retrain the Alert Model or Classifier h: both are used as the dataset ships them. By default it does not re-run the choice of the router's settings either (see the switch below).

**Permission to download.** The next cell asks for your permission to download FiFAR from figshare. Nothing is downloaded until you give it.

In [ ]:
# Settings
FIFAR_DOWNLOAD_ALLOWED = False   # set to True to give permission to download FiFAR from figshare (CC BY 4.0);
                                 # nothing is downloaded, and the run stops at the set-up, until it is True

**The switch** (its own cell):

In [ ]:
# HYPERPARAM_TUNE
# False (default): the router's 250 models are refitted by the framework's own
#   procedure from the settings the study's settings search chose
#   (results/router_selected_settings.csv). This reproduces the study's figures.
# True: the framework's full settings search runs first, 120 trials per model,
#   then the run continues as normal. About 7 hours on a typical laptop;
#   14 to 21 hours on free Colab, which is more than one free session.
#   The results should then equal the study's within the tolerance stated below.
HYPERPARAM_TUNE = False

**What a re-run gives.** Every figure equal to the study's within |x − y| ≤ 10⁻⁵ × max(1, |y|), where y is the study's figure, and the same verdicts. Not byte-identical files: another processor and other library builds change the order of floating-point sums.

In [ ]:
# The repository: this notebook's own folder when it is run from a clone of SaveHITL-pub; otherwise (Colab) a fresh
# clone. Then the run's layout and the state of the switch.
import os, sys, subprocess
REPOSITORY = "https://github.com/zsipka/SaveHITL-pub.git"
if not os.path.isdir("savehitl"):
    if not os.path.isdir("SaveHITL-pub"):
        subprocess.run(["git", "clone", "-q", REPOSITORY, "SaveHITL-pub"], check=True)
    os.chdir("SaveHITL-pub")
sys.path.insert(0, os.getcwd())
from savehitl import notebook as nb
nb.start(HYPERPARAM_TUNE);

In [ ]:
# Environment (Python 3.7.9 and the pinned versions), the framework at its pinned commit with the four patches, and
# FiFAR from figshare; every file is stopped on its sha256. Each part runs once per session.
nb.setup(FIFAR_DOWNLOAD_ALLOWED)

## The heavy steps

Five steps take most of the run's time and memory: regenerating the 50 reviewers; building the router's training files for the five teams; fitting the router's 250 models (a refit by default, the full settings search with the switch); routing the alerts under analyst-aware deferral; and re-drawing the cases to give each difference its range, with the verdicts. The time and peak memory of every step in this run are in the table "Run time and memory of this run" in the technical section.

## How to read the results

- **Alerts.** The cases are the 4,457 bank account applications of the test month that the dataset's fraud model (the **Alert Model**) flagged as suspicious: 711 are frauds and 3,746 are legitimate. Every alert must be decided. **Confirming** an alert rejects the application; **dismissing** it accepts the application.
- **The classifier.** The project outline's "the model" is the framework's **Classifier h**, a gradient-boosted decision tree model that decides each alert on its own.
- **Reviewers.** The simulated analysts are the framework's **Synthetic Experts**, regenerated for this study (section 5). Teams have ten reviewers.
- **Capacity** is the share of the 4,457 alerts a team can review, from 0 (none) to 1 (all). The **realistic range** is 0.65 to 1, which includes 10/11, the share the benchmark gives its own reviewers.
- **Cost ratio** is the cost of wrongly rejecting a legitimate applicant divided by the cost of missing a fraud. The main results use 0.057, the ratio Classifier h was trained for; 0.035, 0.0537 and 0.171 are also shown.
- **Three measures, never combined.** *Expected loss*: each missed fraud counts 1 and each wrongly rejected legitimate applicant counts the cost ratio, summed over the 4,457 alerts. *Regulatory exposure*: the number of frauds accepted. *Fairness exposure*: the age gap, the false rejection rate of legitimate applicants aged 50 or over minus that of applicants under 50. On all three, lower is better; for the age gap, the verdict is on its size, so closer to zero is better.
- **Differences** are always "the policy named first minus the one it is compared with", on the same alerts. A negative difference favours the policy named first.
- **Verdicts.** Each difference comes with a range that shows how much it could vary had other cases fallen in the test month (the 4,457 cases are re-drawn 50,000 times; 1,000 times for full automation on its own). ✅ **beats**: the whole range is below zero. ❌ **does worse**: the whole range is above zero. 🟡 **no difference shown**: the range includes zero. ✅*, ❌* and 🟡* mark a cell that is also a deciding result. The symbols are used in every table with a verdict column.
- **Deciding and descriptive results.** A small set of comparisons was named in the design as the ones that decide the hypothesis. Their ranges are widened so that, across all of them together, the chance of a false finding stays near 5%. Every other result is *descriptive*: it uses an ordinary 95% range, decides nothing, and may differ from a deciding verdict on the same point. Where the two differ, the deciding verdict governs.
- **Versions and random orders.** Each team is run in five versions. The five versions are five disjoint sets of ten reviewers, drawn at random from the 50 with a fixed seed; each reviewer is in exactly one. The notebook repeats this draw, and stops if it does not give the teams for which the router's training files are built. A team **beats** or **does worse** if all five versions do; otherwise its verdict is that of the average of the five versions. Each version is also run with five random orders in which the alerts are taken; the results are averaged over them, and their range is shown.
- **"By construction"** marks a figure that follows from the rules themselves. It is a check, never a finding.
- **A dash (—)** in a table marks an empty cell: the figure or note does not apply to that row.

## 1. Problem

The project outline starts from the observation that human oversight is assumed to reduce risk by default, while in practice reviewers may face more alerts than they can handle, review cases the model already gets right, or defer to the model. The study asks the operational question directly: in a simulated fraud review, when does sending an alert to a human reviewer lower risk, and when does it not?

## 2. Objective

The objective was to define "appropriately applied" human-in-the-loop (HITL) in measurable terms and to test it on a fraud-review use case. The study measures risk on three axes (section 6), compares three ways of deciding alerts (section 3), and judges every comparison by rules fixed in the study's design (section 8).

The four proposed criteria (section 4) are not tested one by one as a definition. Two of them are built into the third of the policies compared (section 3), one is measured, and one can be observed only indirectly. Section 4 says which.

## 3. Hypothesis

**What "deferral" means in this study.** The study replaces calibrated deferral with analyst-aware deferral (one of the aspirational extensions) as it better illustrates the objective.

The hypothesis has two parts. **First**, across realistic review capacity, analyst-aware deferral achieves lower risk than both full automation and naive review of every alert. **Second**, naive review can be worse than full automation when capacity is exceeded or when reviewers largely agree with the model.

**The three policies**, named by number in the tables:
- **policy 1, full automation**: Classifier h decides every alert;
- **policy 2, naive review**: every alert is meant for review; the team reviews as many as its capacity allows, and every alert beyond capacity is confirmed without review, so the application is rejected (i.e. zero regulatory exposure for alerts beyond capacity);
- **policy 3, analyst-aware deferral**: each alert goes to whichever decision-maker, one of the reviewers or Classifier h, is estimated to be most likely to decide it correctly, as long as review capacity lasts; every alert not sent to a reviewer is decided by Classifier h. The estimates come from the framework's **OvA Classifiers**, one gradient-boosted model per reviewer, here called the router.

**The three teams** (section 5) are A, a standard mixed team; B, a team that agrees with the model; and C, a team biased against applicants aged 50 or over.

Each part of the hypothesis is judged separately on each of the three measures. The comparisons below are at the cost ratio 0.057. How they turn into a verdict on the hypothesis is in section 8.

The comparisons are named:
- *team A at full capacity against full automation*;
- *team B against full automation, at each realistic capacity*;
- *team C against full automation, at each realistic capacity*;
- for naive review only: *team B against team A, both at capacity 0.75*, and *team C against team A, both at capacity 0.75*;
- for analyst-aware deferral only: *team A at full capacity against naive review*, *team B at capacity 0.75 against naive review* and *team C at capacity 0.75 against naive review*.

Capacity 0.75 is where capacity counts as materially exceeded.

### 3.1 Naive review against full automation: the results that decide

**Naive review: the results that decide, at cost ratio 0.057.** The graph shows every deciding result of naive review in one grid: one row per measure (for the age gap, its size), one column per team, capacity across, and the difference from full automation with its widened range. The colour gives the verdict. In the panels of teams B and C, a second marker shows the team against team A, both at capacity 0.75. Two results follow from team C's rule and are not findings; they are drawn hollow, and the signed age gap is labelled "signed". The results are also in the files `tables/P2-T3_deciding_results.csv` (every deciding result), `tables/P2-T3b_realistic_range_verdicts.csv` (the verdicts at each realistic capacity) and `tables/P2-T3c_rule_outcomes.csv` (what the rules conclude). The full table and a graph of each result with its range are in the technical section.

The tables and graphs this notebook produces are computed from FiFAR (Alves et al., figshare, 2025, https://doi.org/10.6084/m9.figshare.28351172, CC BY 4.0, https://creativecommons.org/licenses/by/4.0/). They are derived from FiFAR by this project, not part of it, and FiFAR's creators have not reviewed or endorsed them.

In [ ]:
nb.section("P2-F10")

(1) These are the results that decide the second part of the hypothesis, and with section 3.2, the refutation conditions.

(2) Team A at full capacity beats full automation on expected loss and on the age gap, and does worse on frauds accepted. Team B shows no difference at most realistic capacities: it beats on expected loss only at the highest capacities, does worse on frauds accepted only at full capacity, and beats on the age gap only at the lowest. Team C beats on expected loss except at the lowest realistic capacities and does worse on the age gap at every realistic capacity. Against team A, team B does worse on expected loss and beats on frauds accepted; team C shows no difference on expected loss, and its two other results are by construction.

(3) The second part is refuted on expected loss and not refuted on frauds accepted or the age gap (section 8.1).

### 3.2 Analyst-aware deferral against full automation and naive review: the results that decide

**Analyst-aware deferral: the results that decide, at cost ratio 0.057.** The graph is drawn as for naive review. A second marker and colour show the three comparisons against naive review: team A at full capacity, and teams B and C at capacity 0.75. The table below it sets both policies' overall false rejection rates beside the age-gap result for teams B and C at capacity 0.75 against naive review, because a smaller age gap can come from rejecting more applicants in both age groups. The results are also in the files `tables/P3-T3_deciding_results.csv` (every deciding result), `tables/P3-T3b_realistic_range_verdicts.csv` (the verdicts at each realistic capacity) and `tables/P3-T3c_rule_outcomes.csv` (what the rules conclude). The full table and a graph of each result with its range are in the technical section.

In [ ]:
nb.section("P3-F10")

Propose to replace with a dot graph of the three policies' overall false rejection rates per team, with the five versions as dots

(1) These are the results that decide the first part of the hypothesis: whether analyst-aware deferral beats full automation at every realistic capacity and naive review in all three comparisons.

(2) Analyst-aware deferral beats full automation on expected loss with every team at every realistic capacity, and beats naive review with teams A and C. It does worse than full automation on frauds accepted with teams A and B at full capacity, and on the age gap with team C throughout.

(3) Against full automation, the first part holds on expected loss and is refuted on frauds accepted and the age gap; against naive review, it is met on no measure (section 8.1).

### 3.3 Where the difference between analyst-aware deferral and naive review comes from

**Analyst-aware deferral minus naive review, expected loss, split into the alerts left to the classifier and the alerts reviewed**

Analyst-aware deferral and naive review differ in two ways at once: which alerts are reviewed and by whom, and what happens to the alerts that are not reviewed (the classifier decides them under analyst-aware deferral; naive review confirms them, rejecting the application). The bars split the expected-loss difference into these two parts, by team and capacity: the alerts analyst-aware deferral leaves to the classifier, and the alerts it sends to a reviewer. The two parts add up to the whole difference.

In [ ]:
nb.section("P3-F4")

(1) It shows whether analyst-aware deferral's difference from naive review comes from the alerts it reviews or from the alerts it leaves to the classifier, which naive review would confirm beyond capacity. Only the first bears on routing to the right reviewer. Descriptive.

(2) In the three deciding comparisons, the advantage on expected loss lies in the alerts reviewed. At the lowest capacities, the part left to the classifier is as large as the reviewed part or larger.

(3) Team B's extra frauds accepted lie in the alerts left to the classifier, so they reflect how the alerts not reviewed are treated, not the routing; team C's lie in the alerts reviewed (section 3.3).

**Teams B and C at capacity 0.75: the difference on the alerts naive review confirms without review**

The table gives, for teams B and C at capacity 0.75, the difference on the alerts naive review confirms without review, which was predicted in direction from the rules. Both are point figures, without a range or a verdict.

Propose to replace with a dot graph of the differences with their range over the random orders, one panel per team, with a line at zero

In [ ]:
nb.section("P3-T8")

**Analyst-aware deferral minus naive review, split into the alerts left to the classifier and the alerts reviewed, on both measures, with ranges and verdicts**

The bars above show only the expected-loss parts, as point values. This table gives the results behind the statements below, for team A at full capacity and teams B and C at capacity 0.75: the part from the alerts analyst-aware deferral leaves to the classifier and the part from the alerts it sends to a reviewer, on expected loss at cost ratio 0.057 and on frauds accepted. Each row is the average of the five versions, with its 95% range and its verdict. These results are descriptive and decide nothing. They are read from the run's full results by their key columns, with nothing recomputed; the cost ratio does not apply to frauds accepted.

Propose to replace with a range graph of each part's difference, coloured by verdict, with a line at zero

In [ ]:
nb.section("rows-split")

- **Team A at full capacity:** on expected loss, analyst-aware deferral beats naive review, and the whole difference lies in the alerts reviewed; on the alerts left to the classifier, no difference is shown. On frauds accepted, neither part shows a difference, and neither does the whole.
- **Team B at capacity 0.75:** on expected loss, the alerts reviewed favour analyst-aware deferral and the alerts left to the classifier show no difference; the whole shows no difference. On frauds accepted, analyst-aware deferral does worse, and the difference lies in the alerts left to the classifier; on the alerts reviewed, no difference is shown.
- **Team C at capacity 0.75:** on expected loss, analyst-aware deferral beats naive review, and the difference lies in the alerts reviewed; the alerts left to the classifier show no difference. On frauds accepted, it does worse, and here the difference lies in the alerts reviewed, not in those left to the classifier.
- **On the alerts naive review confirms without review** (teams B and C at 0.75, the first table above): naive review accepts no fraud on these alerts, by its rule, so analyst-aware deferral accepts as many frauds or more on them. On expected loss, these alerts were predicted to favour analyst-aware deferral. The prediction held for both teams: averaged over the random orders, analyst-aware deferral has the lower expected loss on these alerts in every version, for team C in every random order and for team B not in every random order (section 8.2).

The verdict of the whole is the deciding verdict of section 3.2; the verdicts of the parts are descriptive.

The split does not separate the router's choice of alerts and reviewers from the way the alerts it does not review are treated. Its results are descriptive and decide nothing; the last table above gives both parts on both measures, with the 95% ranges and verdicts behind these statements.

## 4. Proposed criteria for appropriate HITL

The project outline proposes that a case should reach a human only if (1) the reviewer is more likely than the model to be correct; (2) the volume routed fits the actual review capacity; (3) the expected loss avoided exceeds the cost of review; (4) the reviewer has the information and time to disagree with the model. In this study:
- **Criterion 1** is built into analyst-aware deferral as the router sees it: a reviewer takes an alert only if the router's estimate of the reviewer being right is strictly above the classifier's confidence. Whether it holds in fact is measured (section 7).
- **Criterion 2** is built in: never more alerts are reviewed than capacity allows.
- **Criterion 3** is measured, not enforced: the loss avoided per review is shown as the cost of one review at which review breaks even (section 7). No cost of review is modelled.
- **Criterion 4** cannot be observed directly in a simulation without time pressure; it shows only through how often reviewers overturn the alert (section 7).

### 4.1 How analyst-aware deferral routed the alerts, and how well the router ranks

**Analyst-aware deferral: how many alerts went to a reviewer, to which reviewer, and to the classifier**

For each team, version and random order, at capacities 1, 10/11, 0.75 and 0.65, the table shows how analyst-aware deferral used the review places: the number of alerts on which the best reviewer outscores the classifier, the share of review places used, how many alerts each of the ten reviewers took and how many the classifier decided, and how often two scores were exactly equal. Teams A and B share one routing. For team C it also shows the alerts that team C's rule leaves to the classifier.

Propose to replace with stacked bars per capacity of the alerts sent to a reviewer, to the classifier and left by team C's rule, with each reviewer's workload as a box plot

In [ ]:
nb.section("P3-T1")

**How well the router tells apart the alerts each reviewer gets right from those the reviewer gets wrong**

Analyst-aware deferral relies on the framework's **OvA Classifiers**, one model per reviewer, here called the router, to estimate how likely each reviewer is to decide an alert correctly. The router's models are trained on the decisions of this study's reviewers by the framework's own procedure, including its settings search; by default this notebook refits them from the settings that search chose (section 5). How well these estimates separate the alerts a reviewer decides correctly from those the reviewer gets wrong is measured as an area under the ROC curve: 0.5 is chance and 1 is perfect. The table shows, for each team version, the average, lowest and highest of this figure over the version's 50 models (its ten reviewers, five training shuffles each), on the test month beside the previous month. The previous month is the one on which the router's models were validated and their settings chosen, so its figures are not measured on unseen data; only the test month's are.

(Each figure is the area under the ROC curve with legitimate applicants weighted 0.057 and frauds 1. The figure for each single model is in the accompanying table file.)

Propose to replace with a dot-and-range graph of the router's ranking quality per version, test month beside previous month, with a line at 0.5

In [ ]:
nb.section("P3-T1r")

Averaged over each version's 50 models, the router's ranking quality is 0.58 to 0.65 on the test month and 0.59 to 0.65 on the previous month; over all 250 models it is 0.61 in both months. Single models range from 0.457 to 0.788 on the test month and from 0.497 to 0.787 on the previous month; 6 of the 250 are at or below 0.5 on the test month, and 1 on the previous month. Because the previous month is the router's validation month, the agreement between the two months says little about how stable the ranking is. The router ranks the reviewers' correctness only weakly. Analyst-aware deferral is therefore tested as the framework implements it, not as the best possible router would do.

## 5. Method

**Data.** The study uses the public FiFAR dataset. Its 30,622 alerts span five months; every result is measured on the test month, and no other month is used for any result.

**Models.** The **Alert Model** and **Classifier h** are used as the dataset ships them; neither is retrained. Classifier h decides by its built-in rule at every cost ratio.

**Reviewers.** The project outline planned to replay the dataset's recorded decisions of its 50 synthetic analysts. The study instead regenerates the 50 reviewers with the dataset's own generator and settings, with two changes: no reviewer gives any weight to the applicant's age, and each is fitted on the four months before the test month instead of one. This makes team C's rule the only age bias the study builds in on purpose. It does not remove every age gap: the regenerated reviewers and the classifier still wrongly reject applicants aged 50 or over more often than those under 50 (sections 5.1 and 6.1). Each regenerated reviewer is built to a target cost per case at cost ratio 0.057, drawn from a distribution with an average of 0.035. Their decisions are fixed, so every policy can be replayed on exactly the same decisions.

**Retraining.** The project outline aimed to minimise retraining. The only models trained are the router's: 250 models, one per reviewer for each of the five teams and five training shuffles. They are trained on the regenerated reviewers' decisions by the framework's own procedure, including its settings search of 120 trials per model. By default this notebook refits them from the settings that search chose; the switch at the top runs the search itself.

**Teams.** In the study, capacity is not a property of a team: every team is run at every capacity from 0 to 1. Within a version, the three teams are made of the same ten reviewers and differ only in how they decide a reviewed alert:
- **Team A, standard mixed team (the control):** members' own decision stands.
- **Team B, agrees with the model:** members follow the model in 4 of 5 reviewed cases, picked at random; otherwise their own decision stands.
- **Team C, biased:** members reject older applicants (aged 50 or over) whenever the model would.

The precise behaviour of each team is fixed by the teams' decision rules in the design.

**What is varied.** Capacity: 0 to 1 in steps of 0.05, plus 10/11 (22 values). Every combination of team, version, random order, capacity and cost ratio is run and reported.

**The automated baseline.** Full automation reproduces the total cost the benchmark publishes for Classifier h on the test month (Alves et al., 2025, p. 24, Table 7).

### 5.1 The reviewers and the teams

The tables describe the teams. For each of the five versions, its ten members; each reviewer is in exactly one version. For the test month at capacities 1 and 0.75: how often team B's decision equals the classifier's, how often team B takes the classifier's decision when the reviewer disagrees, how often the reviewers themselves disagree with the classifier, and how many decisions team C's rule changes.

In [ ]:
nb.section("P2-T1")

Each regenerated reviewer has a target: the rate of wrongly rejected legitimate applicants, the share of frauds let through and the cost per case it was built to reach. The regenerated figures are those it actually reaches, on the four months it was fitted on and on the month before the test month alone. Both, per team version and per reviewer, with each reviewer's weights on the alert score and on age, are in `tables/P2-T1c_team_version_rates.csv` and `tables/P2-T1d_twins.csv`.

In the test month, team B's decision equals the classifier's on 0.90 to 0.93 of the reviewed alerts, at full capacity and at 0.75 (range over the five versions); where the reviewer disagrees with the classifier, team B takes the classifier's decision in 0.79 to 0.80 of cases, as its rule sets. Team C's rule changes, per version and averaged over the random orders, 378 to 620 decisions on legitimate applicants and 13 to 30 on frauds at full capacity, and 288 to 476 and 10 to 24 at capacity 0.75.

## 6. Risk measures

The three measures are those of the project outline, made precise:
- **Expected loss**, with the cost ratio set and varied as the project intended: four values, with 0.057, the ratio Classifier h was trained for, as the main one. Results say nothing about cost ratios below 0.035 or between 0.035 and 0.0537.
- **Regulatory exposure** is measured as the number of frauds accepted among the flagged applications, with the share of frauds let through. Frauds the Alert Model did not flag are outside the study.
- **Fairness exposure** is the gap in false rejections between applicants aged 50 or over, the framework's declared protected group, and those under 50.

The graphs in this section are descriptive: they show the full range of capacities, with ordinary 95% ranges. The deciding verdicts are those of section 3.

### 6.1 Full automation: the baseline

**Full automation and "confirm every alert": the three measures at each cost ratio**

Full automation is the baseline every other policy is compared with, alert by alert. The table gives its three measures at each cost ratio, each with a 95% range from 1,000 re-draws of the cases. Beside it is a reference figure, not a policy: "confirm every alert", where every flagged application is rejected.

In [ ]:
nb.section("P1-T2")

**Full automation and "confirm every alert": the three measures against the cost ratio**

The graph draws the same figures against the cost ratio, with 0.057 marked. Away from 0.057 the classifier's decisions are kept as they are, not re-optimised for the cost ratio.

In [ ]:
nb.section("P1-F1")

(1) Full automation is the reference for every comparison. The graph shows how its expected loss moves with the cost ratio while its decisions stay the same. Descriptive.

(2) Full automation's frauds accepted and age gap do not change with the cost ratio, because its decisions do not; its expected loss rises with the ratio. Confirming every alert has the lower expected loss only at the lowest cost ratio.

(3) It sets the reference and decides nothing.

### 6.2 Expected loss against capacity

**Naive review minus full automation, expected loss, against capacity**

Each panel is one team. The thin lines are the five versions, the thick line their average with its 95% band; the realistic range, 0.65 to 1, is shaded, and the points where the average crosses zero are marked. Below zero the policy has the lower expected loss.

In [ ]:
nb.section("P2-F1")

(1) It shows over the whole capacity grid where naive review's expected loss lies against full automation, including capacities outside the realistic range. Descriptive.

(2) No team does worse than full automation at any capacity; on the average of the five versions, each team beats it from some capacity upwards: team A first, then team C, and team B only at the higher capacities.

(3) On expected loss naive review never falls below full automation, which refutes the second part on this measure (section 8.1).

**Analyst-aware deferral minus full automation, and minus naive review, expected loss, against capacity**

For analyst-aware deferral, blue is the difference from full automation and red the difference from naive review.

In [ ]:
nb.section("P3-F1")

(1) As P2-F1, for analyst-aware deferral, against both other policies. Descriptive.

(2) Analyst-aware deferral beats full automation for every team from the lowest capacities above zero, and beats naive review at every realistic capacity on the ordinary ranges.

(3) It supports the first part on expected loss against full automation; against naive review, the deciding result for team B shows no difference (section 3.2).

**Naive review minus full automation, expected loss, split into the reviewed alerts and the alerts beyond capacity**

Naive review can differ from full automation for two reasons: on the alerts the team reviews, the team decides instead of the classifier; on the alerts beyond capacity, every alert is confirmed instead. The bars split the expected-loss difference into these two parts, by team and capacity, as point values (average of the five versions). The part beyond capacity is the same for teams B and C, which leave the same alerts unreviewed.

In [ ]:
nb.section("P2-F4")

(1) It separates what the reviewers' own decisions contribute from what the confirmation of alerts beyond capacity contributes, which is the mechanism behind "capacity exceeded". Descriptive.

(2) The prediction that the alerts beyond capacity raise expected loss held in point value at the realistic capacities except 0.95; the alerts beyond capacity are not shown to change expected loss in either direction. The reviewed alerts lower it, by much more for team C than for team B.

(3) In this study, exceeded capacity does not by itself make naive review worse than full automation on expected loss.

### 6.3 Frauds accepted against capacity

**Naive review minus full automation, frauds accepted, against capacity**

The same layout as for expected loss, for the number of frauds accepted. At capacity 0, naive review confirms every alert and so accepts no fraud; analyst-aware deferral is full automation.

In [ ]:
nb.section("P2-F2")

(1) Frauds accepted is the regulatory measure. The graph shows where naive review accepts more or fewer frauds than full automation across capacity. Descriptive.

(2) At low capacity naive review accepts fewer frauds than full automation, because the alerts beyond capacity are confirmed; as capacity rises the reviewers' own decisions take over, and on the ordinary ranges, at full capacity every team accepts more frauds than full automation (team A over most of the realistic range). For team C at full capacity, the deciding verdict shows no difference (sections 3.1 and 6.5).

(3) On this measure naive review can fall below full automation, through the reviewers' own decisions rather than through exceeded capacity (section 8.1).

**Analyst-aware deferral minus full automation, and minus naive review, frauds accepted, against capacity**

In [ ]:
nb.section("P3-F2")

(1) As P2-F2, for analyst-aware deferral, against both other policies. Descriptive.

(2) On the ordinary ranges, analyst-aware deferral accepts more frauds than full automation with every team at every realistic capacity, and more than naive review except at the highest capacities, where team A stops sooner than teams B and C.

(3) On frauds accepted the first part is refuted (section 8.1).

### 6.4 Age gap against capacity

**Naive review minus full automation, size of the age gap, against capacity, with the overall false rejection rate**

The same layout, for the size of the age gap, with the policy's overall false rejection rate on a second axis. The second axis matters for naive review: alerts beyond capacity are rejected in both age groups alike, which pulls the gap towards zero. Where naive review's gap is smaller while it rejects more applicants overall, the study reports "gap narrowed by rejecting more applicants in both groups", not a fairness improvement.

In [ ]:
nb.section("P2-F3")

(1) The age gap is the fairness measure. The overall false rejection rate is shown beside it because a gap can narrow when more applicants of both groups are rejected. Descriptive.

(2) Teams A and B narrow the age gap against full automation at almost every capacity. At low capacities this comes with a higher overall false rejection rate than full automation's, in point value, because the alerts beyond capacity are confirmed. In the realistic range this concerns team B only, at its lowest capacities; at the highest of these the excess is marginal and lies within the range of team B's own rate. Team C widens the gap from low capacities upwards, by its rule.

(3) The age-gap verdicts that decide are in section 3.1.

**Analyst-aware deferral minus full automation, and minus naive review, size of the age gap, against capacity, with the overall false rejection rate**

In [ ]:
nb.section("P3-F3")

(1) As P2-F3, for analyst-aware deferral, against both other policies. Descriptive.

(2) Against full automation, team A narrows the gap, team B shows no difference and team C widens it at every realistic capacity; against naive review, teams B and C have the larger gap at most realistic capacities.

(3) On the age gap the first part is refuted (section 8.1).

### 6.5 Where each team crosses full automation, per measure

**Naive review: where each team's difference from full automation crosses zero, and the verdict at each capacity**

These tables give the numbers behind the crossings described above, for each team and measure: the capacity where the average difference crosses zero, and the descriptive verdict at every capacity of the grid. They are descriptive. Where a cell at a realistic capacity is also a deciding result, the deciding verdict in section 3 governs.

Propose to replace with a colour grid of the verdicts at each capacity, one panel per measure, for both tables below

In [ ]:
nb.section("P2-T4")

**Analyst-aware deferral: where each team's difference from full automation and from naive review crosses zero, and the verdict at each capacity**

In [ ]:
nb.section("P3-T4")

The crossings and verdicts in these tables are those described under each measure in sections 6.2 to 6.4. On the average of the five versions, some descriptive verdicts at realistic capacities differ from the deciding verdicts of section 3; the list below is complete for the average only, and single versions differ in more places (the rows of each version are in the table files). Under naive review, against full automation: team B on expected loss at 0.85, 0.90 and 10/11, on frauds accepted at 0.65 and 0.70, and on the age gap at 0.70 to 0.95 (descriptive "beats", deciding "no difference shown"); team C on expected loss at 0.65 and 0.70 (descriptive "beats") and on frauds accepted at 0.95 and 1 (descriptive "does worse"), where the deciding verdict is "no difference shown". Under analyst-aware deferral: against full automation, on frauds accepted, team B at 0.65 to 0.95 and team C at 0.65 to 1 (descriptive "does worse", deciding "no difference shown"); against naive review, team B at 0.75 on expected loss (descriptive "beats") and on the age gap (descriptive "does worse"), where the deciding verdict is "no difference shown". In each case, and for every single version, the deciding verdict governs. Two readings are not supported: a crossing of zero is not a refutation on its own, and crossings at cost ratios other than 0.057 are not given here (section 6.8). The grid steps by 0.05, so a crossing is located only to within that step. A summary that reads "does worse up to capacity X; never beats" can be misread; the caveat in the technical section explains it.

### 6.6 Does team C's bias grow or shrink when capacity is exceeded?

**Naive review: false rejection rates by age group for teams C and A against capacity, and how team C's bias changes between capacity 0.75 and full capacity**

The project outline's question is whether a bias compounds or cancels when capacity is exceeded. The measure is the difference between teams C and A at capacity 0.75, minus the same difference at full capacity, on three figures: the signed age gap, the number of wrongly rejected applicants aged 50 or over, and expected loss. Above zero, the bias compounds with overload; below zero, it cancels in part; otherwise it is additive. As for every difference, a verdict of "beats" on this change means only that its whole range is below zero, and "does worse" that it is above zero. For naive review, the graph also shows both teams' false rejection rates by age group against capacity, and team C's gap over team A against the line expected if the bias simply scaled with capacity.

In [ ]:
nb.section("P2-F5")

(1) The project outline links bias to exceeded capacity. The graph shows whether team C's bias grows or shrinks as capacity falls, against the line expected if it scaled with capacity. Descriptive.

(2) Team C's excess over team A on the age gap shrinks, not grows, as capacity falls from full to materially exceeded, because every team confirms the alerts beyond capacity; this follows from the rules. On expected loss the two effects cancel in part.

(3) It gives no support to bias growing with exceeded capacity, and says nothing about real reviewers.

**Analyst-aware deferral: how the difference between teams C and A changes between capacity 0.75 and full capacity**

The table gives the same measure for analyst-aware deferral, but there it measures something else. Team C's routing is the same at 0.75 and at full capacity, so the measure reduces to team A's own change between the two capacities and says nothing about team C's bias, although the rows are labelled team C. Its readings ("cancels (in part)", "additive"), the kind "finding" and its verdicts describe team A's change only.

In [ ]:
nb.section("P3-T7")

- **Naive review.** Team C's excess over team A, on the signed age gap and on the number of wrongly rejected applicants aged 50 or over, is smaller at capacity 0.75 than at full capacity, in all five versions. This direction follows from the rules and is a check, not a finding: team C's age-gap difference from full automation grows with capacity, and team C does worse on the age gap even at full capacity, where nothing is beyond capacity. The one finding is on expected loss: its verdict is "beats" and its reading "cancels (in part)"; three of the five versions read "cancels (in part)" and two "additive". The line expected if the bias scaled with capacity supports no verdict.
- **Analyst-aware deferral.** Team C's routing is the same at capacity 0.75 and at full capacity. The measure therefore reduces to team A's own change between the two capacities: it says nothing about team C's bias, and its readings describe team A only. On the signed age gap, no difference is shown and the reading is "additive"; on the number of wrongly rejected applicants aged 50 or over and on expected loss, the verdict is "beats" and the reading "cancels (in part)", in all five versions.

Neither result shows that overload degrades reviewers' judgement or that a bias in real reviewers acts this way.

### 6.7 Who gets a reviewer: access by age group

**Share of alerts sent to a reviewer, by age group, analyst-aware deferral beside naive review**

This table and graph ask who gets access to a human reviewer. The alerts fall into four groups: applicants aged 50 or over or under 50, and alerts the classifier would confirm (rejecting the application) or would dismiss. For each group, the share of alerts sent to a reviewer is shown (i) overall, (ii) among the alerts the classifier would confirm, and (iii) among those it would dismiss, with the difference between the age groups in percentage points. Analyst-aware deferral is shown beside naive review at the same capacity. The figures are point figures, pooled over the five versions and five random orders, without a range or a verdict. Some cells follow directly from the rules and are labelled "by construction".

In [ ]:
nb.section("P3-T9")

**Share of alerts sent to a reviewer against capacity, by age group, analyst-aware deferral and naive review**

The graph draws shares (i) and (ii) at every capacity: solid lines for analyst-aware deferral, dashed for naive review.

In [ ]:
nb.section("P3-F9")

(1) It shows whether older and younger applicants get the same access to a reviewer under each policy, which bears on fairness exposure beyond the outcome. Descriptive.

(2) For teams A and B, both age groups reach a reviewer in nearly the same share under both policies. Under analyst-aware deferral with team C, applicants aged 50 or over reach a reviewer far less often than those under 50, and none whose alert the classifier would confirm, by its rule; under naive review, team C's two age groups reach a reviewer in the same share.

(3) Equal outcomes on average can hide unequal access; this decides nothing.

### 6.8 Other cost ratios

**Naive review: expected-loss verdict against full automation at each cost ratio and capacity**

Each grid shows, for each team, the descriptive expected-loss verdict against full automation at every capacity (across) and cost ratio (down): green beats, red does worse. Away from 0.057, neither full automation nor the reviewers (nor, for analyst-aware deferral, the router) is re-tuned for the cost ratio, so every "beats" there holds only "against an automation baseline not re-optimised for this cost ratio". The table labels each team and capacity: **"conditional"** if it beats full automation at one or more of the other cost ratios and does worse at none; **"not robust"** if it beats at one or more and does worse at one or more. An empty label means the team beats full automation at none of the other cost ratios; it does not mean "no difference".

In [ ]:
nb.section("P2-F7")

(1) The cost ratio is a parameter of the study. The graph shows whether naive review's verdict on expected loss holds away from the main ratio, against a baseline not re-tuned for it. Descriptive.

(2) Where a team beats full automation at the main cost ratio, it also beats at the highest ratio and, with one exception for team A, at the ratio just below the main one; at the lowest ratio most of these turn to no difference, and at the highest every team does worse at low capacities.

(3) No naive-review result is shown to be robust across cost ratios. Even where a team beats full automation at all four ratios (team C at the higher capacities), the result is "conditional", because full automation is not re-tuned for each ratio (section 6.8).

**Analyst-aware deferral: expected-loss verdict against full automation at each cost ratio and capacity**

In [ ]:
nb.section("P3-F6")

(1) As P2-F7, for analyst-aware deferral. Descriptive.

(2) Teams A and C beat full automation at every capacity above 0 at all four cost ratios; team B does so only from a middle capacity upwards; below it, team B shows no difference at the lowest cost ratio, and at the lowest capacity above zero it beats only at the highest ratio.

(3) Every result is "conditional": none is shown against a baseline re-tuned for the cost ratio (section 6.8).

**Against "confirm every alert" at cost ratio 0.035: the results quoted below**

At cost ratio 0.035, confirming every alert has a lower expected loss than full automation in point value (section 6.1), so beating full automation there does not mean beating the cheaper automated option. The first table gives naive review's comparison with confirming every alert for team C at capacity 0.15; the second gives analyst-aware deferral's for teams A, B and C at capacities 0.75 and 1. Each row is the expected-loss difference, the policy minus confirming every alert, averaged over the five versions, with its 95% range and its verdict. These results are descriptive and decide nothing. They are read from the run's full results by their key columns, with nothing recomputed.

Propose to replace with a range graph of the differences from confirming every alert, coloured by verdict, with a line at zero, for both tables below

In [ ]:
nb.section("rows-0035")

- **Naive review.** Team A is labelled "conditional" from capacity 0.45 to 1 and team B from 0.80 to 1; team C is labelled "not robust" at 0.15 to 0.25 and "conditional" at 0.40 and from 0.55 to 1. The rows left empty are team A at 0 to 0.40, team B at 0 to 0.75, and team C at 0 to 0.10, 0.30, 0.35, 0.45 and 0.50; the team does worse at 0.171 in 7 of team A's 9 empty rows, 14 of team B's 16 and 5 of team C's 7. At 0.171, each team does worse at low capacities where it shows no difference at 0.057: team A at 0 to 0.30, team B at 0 to 0.65 and team C at 0 to 0.35. At 0.035, most results that beat at 0.057 show no difference: all of team A's and team B's, and 4 of team C's 11 (0.55 to 0.70); team C still beats at 0.75 to 1. Where a verdict changes, the break-even lies between 0.057 and the tested ratio.
- **At 0.035**, confirming every alert has the lower expected loss of the two automated options in point value (section 6.1; a reference, not a finding). Under naive review, only team C beats full automation at 0.035, at capacities 0.15 to 0.25, 0.40 and 0.75 to 1; at 0.15 it also beats confirming every alert (the first table above).
- **Analyst-aware deferral.** Every team is labelled "conditional" at every capacity above 0; no result is "not robust". At 0.035, team A beats full automation at every capacity from 0.05 but does not also beat confirming every alert at 14 of them (0.05 to 0.70); team B beats full automation at 14 capacities (0.40 to 1) and confirming every alert at none of them; team C beats full automation at every capacity from 0.05 but does not also beat confirming every alert at 10 of them (0.05 to 0.50). Teams A and C also beat confirming every alert at 0.75 and 1; team B does not (the second table above).

Away from 0.057, neither full automation, the reviewers nor the router is re-tuned for the cost ratio, so no result here shows a team beating full automation robustly across cost ratios. The results against confirming every alert are in the two tables just above, with their ranges and verdicts.

## 7. Simple governance effectiveness monitoring

The project outline proposes four indicators a bank could compute from its own review logs: reviewer override rate, capacity utilisation, loss avoided per review, and subgroup disparity. The graphs show them for each team against capacity, averaged over the five versions, with 95% bands where a range exists. They are not defined at capacity 0, where nothing is reviewed. Two are adapted. Under naive review every review place is always used, so capacity utilisation is shown as the number of alerts per review place (1 ÷ capacity, the same for every team); under analyst-aware deferral, as the share of review places used. Loss avoided per review, against full automation, is the cost of one review at which review breaks even, in units of a missed fraud (criterion 3). For analyst-aware deferral, two more panels show criterion 1 as it turns out: on reviewed alerts, the share where the reviewer is right and full automation wrong, and the reverse.

In [ ]:
nb.section("P2-F6")

(1) These are the indicators the project outline proposes for detecting inappropriate oversight from review logs. No rule was fixed for them; they decide nothing.

(2) Team A overturns the most reviewed alerts, team B the fewest and team C in between, almost independently of capacity; team B's loss avoided per review is about a fifth of team A's; team C's age gap on reviewed alerts is much the largest.

(3) The indicators separate the three teams clearly, but no rule turns them into a verdict.

**Analyst-aware deferral: monitoring indicators against capacity, per team, with criterion 1 as it turns out**

In [ ]:
nb.section("P3-F5")

(1) As P2-F6, and it shows criterion 1 as it turns out: how often the reviewer is right where the classifier is wrong, and the reverse. Descriptive.

(2) Teams A and B use every review place except near full capacity, where they use slightly fewer; team C uses fewer from a lower capacity, because fewer of its alerts are routed to a reviewer. On reviewed alerts, the reviewer is right where the classifier is wrong far more often than wrong where the classifier is right.

(3) Criterion 1 holds on the reviewed alerts as it turns out; the indicators decide nothing.

No rule was fixed for these indicators, so they carry no verdict and support no conclusion about detecting inappropriate oversight before it fails. They show what such indicators look like in this simulation. Criterion 4 shows only through the override rate. Two readings follow from the rules, and both hold in this run: under naive review, the load is the same for every team at a given capacity, and team B's loss avoided per review is about a fifth of team A's.

## 8. Integrity of the result

As the project outline requires, every combination is reported, including those unfavourable to the hypothesis.

The project outline gives two conditions under which the hypothesis is refuted. The study applies each to every measure separately:
- **First condition: "deferral fails to outperform automation at any capacity level".** Read strictly: if analyst-aware deferral fails to beat full automation in at least one of its deciding results against full automation (team A at full capacity; teams B and C at each realistic capacity), the hypothesis is refuted on that measure.
- **Second condition: "naive HITL never falls below automation".** If no team under naive review does worse than full automation in any of its deciding results against full automation, the hypothesis is refuted on that measure.

The second part of the hypothesis is judged with two further rules: naive review with capacity exceeded (team C below full capacity) and naive review by reviewers who agree with the model (team B). The first part is also judged against naive review: analyst-aware deferral must beat naive review in all three comparisons on a measure.

### 8.1 The hypothesis, measure by measure

The rule-outcome files `tables/P2-T3c_rule_outcomes.csv` and `tables/P3-T3c_rule_outcomes.csv` apply the rules of the design to the deciding results of section 3, one measure at a time. Each row is one rule and one measure, with its outcome and the teams and capacities that produce it.

**Expected loss.** Analyst-aware deferral beats full automation in every one of its deciding results against it, so the first refutation condition is not met. Against naive review it beats with team A at full capacity and team C at 0.75, but shows no difference with team B at 0.75, so the rule that it beats naive review in all three comparisons is not met. Under naive review, no team does worse than full automation in any deciding result: team A at full capacity beats it, team B shows no difference up to 10/11 and beats at 0.95 and 1, and team C shows no difference at 0.65 and 0.70 and beats from 0.75. The second refutation condition is therefore met: on expected loss, the hypothesis is refuted. Neither the rule for exceeded capacity (team C) nor the rule for reviewers who agree with the model (team B) is supported. Team C's bias rule shows no difference in expected loss at 0.75 against the same reviewers without it.

**Frauds accepted.** Analyst-aware deferral does worse than full automation with teams A and B at full capacity and shows no difference elsewhere, so the first refutation condition is met: on frauds accepted, the hypothesis is refuted. Against naive review, it shows no difference with team A and does worse with teams B and C at 0.75. Under naive review, teams A and B at full capacity do worse than full automation, where the reviewers themselves do worse than the classifier on the alerts they review, so the second refutation condition is not met. The rule for reviewers who agree with the model is supported on this measure (team B at full capacity), but not through the agreement. The cause is that the reviewers themselves do worse than the classifier on the alerts they review. Agreeing with the model moves team B towards the classifier: at full capacity, team B's excess of frauds accepted over full automation is about a fifth of team A's (section 8.2). The rule for exceeded capacity is not supported (team C shows no difference at every realistic capacity).

**Size of the age gap.** Analyst-aware deferral beats full automation with team A at full capacity, shows no difference with team B and does worse with team C at every realistic capacity, so the first refutation condition is met: on the age gap, the hypothesis is refuted. Against naive review, it shows no difference with teams A and B and does worse with team C at 0.75. Under naive review, team C does worse than full automation at every realistic capacity, so the second refutation condition is not met, and the rule for exceeded capacity is supported; team C's age gap comes from its rule (section 6.6). The rule for reviewers who agree with the model is not supported: team B beats full automation at 0.65 and shows no difference elsewhere.

None of this shows what real reviewers, a better router or a cost of review would give.

### 8.2 Predicted from the rules, and how they turned out

Several outcomes were predicted from the rules. All are reported, whether they held or not:
- **Team B moves only about a fifth of the way from full automation**, because it takes the model's decision on most alerts where the reviewer disagrees. Held: at full capacity, team B's difference from full automation is 0.20 of team A's on expected loss and 0.21 on frauds accepted.
- **Under naive review, the alerts beyond capacity raise expected loss at every realistic capacity below full.** Held in point value at the realistic capacities except 0.95; the alerts beyond capacity are not shown to change expected loss in either direction.
- **Team B against team A at 0.75 takes the opposite direction to team A against full automation.** Held on expected loss and on frauds accepted: team B does worse than team A on expected loss and beats it on frauds accepted, the opposite of team A against full automation; on the age gap, no difference is shown. This is a consistency reading, not a check, because team A's comparison is at full capacity.
- **Team C accepts no more frauds than team A, and its signed age gap is at least team A's**, in every random order and version. Held in every random order of every version at all 22 capacities (110 of 110 version and capacity rows).
- **On the alerts naive review confirms without review**, analyst-aware deferral accepts as many frauds or more and has the lower expected loss. Held for teams B and C: it accepts more frauds on these alerts and, averaged over the random orders, has the lower expected loss in every version (section 3.3).
- **Under analyst-aware deferral, team C sends no alert of an applicant aged 50 or over that the classifier would confirm to a reviewer, and team B's access figures equal team A's.** Held in every version, random order and capacity (section 6.7).

### 8.3 The full sweep

**Naive review: every team, version, capacity and cost ratio, all measures**

The project outline requires the full sweep to be reported, including settings in which naive review performs acceptably. These tables show the differences from full automation (and, for analyst-aware deferral, from naive review) at cost ratio 0.057 for every team and capacity, with the 95% range, the descriptive verdict and the range over the five random orders. In these tables, "seeds" means the five random orders. Every team version, cost ratio and further measure is in the full table files that accompany the notebook.

In [ ]:
nb.section("P2-T5")

**Analyst-aware deferral: every team, version, capacity and cost ratio, all measures**

In [ ]:
nb.section("P3-T5")

Nothing in the sweep is left out: every team, version, capacity, cost ratio and measure that was run is in these tables or in their table files, and every row not discussed above is descriptive and decides nothing. These tables hold the differences from full automation and, for analyst-aware deferral, from naive review. Full automation's own figures (section 6.1), the deciding results with their widened ranges (sections 3.1 and 3.2, and in full in the technical section), the figures on the alerts naive review confirms without review (section 3.3) and the shares of alerts sent to a reviewer (section 6.7) are in their own tables.

### 8.4 How many differences chance alone would produce

**Naive review: observed "beats" and "does worse" against the number expected by chance**

With so many descriptive comparisons, some would show a difference by chance alone. Each graph sets the number of descriptive "beats" and "does worse" verdicts against full automation against the number expected by chance (2.5% on each side). Neighbouring capacities are not independent, so this is a guide, not a test.

The two results fixed by construction compare two teams, so none is in the counted set; this exclusion removes nothing.

In [ ]:
nb.section("P2-F8")

(1) With many descriptive ranges, some differ from zero by chance alone. The graph compares the observed count with that number, so that the descriptive results are not over-read. Descriptive.

(2) Both "beats" and "does worse" occur far more often than the number expected by chance.

(3) The comparison is a guide to reading, not a test: the descriptive ranges share the same re-draws and overlap across capacities, versions and the average of the five, so the number expected by chance does not say how rare the observed counts would be. Each single descriptive result decides nothing.

**Analyst-aware deferral: observed "beats" and "does worse" against the number expected by chance**

Capacity 0 is left out entirely, because there analyst-aware deferral is full automation.

In [ ]:
nb.section("P3-F7")

(1) As P2-F8, for analyst-aware deferral. Descriptive.

(2) As for naive review, both counts are far above the number expected by chance.

(3) As for P2-F8: a guide to reading, not a test.

## 9. Limitations

- **The reviewers are simulated.** Each makes one fixed decision per alert; there is no fatigue, time pressure or learning, and exceeding capacity changes only what happens to the alerts beyond it. The project outline's criterion 4 shows only through the override rate.
- **No cost of review is modelled.** Loss avoided per review is shown as the break-even cost instead.
- **One alert model, one test month:** 4,457 alerts, 711 of them frauds. Frauds the Alert Model did not flag are outside the study.
- **The cost ratio is implied, not observed.** The data suggest a ratio of roughly 0.03 to 0.045; of that range only 0.035 is tested. Away from 0.057, neither full automation, the reviewers nor the router is re-tuned, and how much better a re-tuned automation would do is not measured.
- **The reviewers are regenerated**, without any weight on age and fitted on four months; they are not the dataset's original reviewers.
- **Team B's 4 in 5** is the author's judgement, not observed behaviour; other values would give other results for team B. **Team C's bias is a rule**, not a model of how bias arises in people.
- **The router is weak.** It ranks the reviewers' correctness only weakly, and some of its models are at or below chance. A router that fits the reviewers less well puts analyst-aware deferral at a disadvantage.
- **The routing's own effect is not separated** from how the alerts not reviewed are treated (section 3.3).
- **Capacity is modelled as one shared pool** of review places, with the classifier unlimited. Workload between reviewers can be very uneven.
- **The alerts are taken in a random order**, not by score, so that the policies are compared like for like; under analyst-aware deferral the order affects which alerts are reviewed, and its effect is part of the spread over the random orders.
- **The ranges cover which cases fell in the test month**, not the policy's own randomness, which is shown as the range over random orders. The rule that a team beats or does worse when all five versions agree, or else by their average, makes the 5% chance of a false finding approximate. The deciding results of the two policies are adjusted separately, so across both together that chance is above 5%.
- **Capacity steps by 0.05**, so every crossing is located only to within that step.
- **The implementation was checked during the study.** The checks are not repeated in this notebook.
- The project outline's own limitation on the author's dual role as project owner and stakeholder stands.

## 10. Risks

The risk named in the project outline, that the pre-trained models might not run well on the available Apple Silicon machine, did not stop any part of the study: the study's own computation ran on the project's machine. The outputs shown in this notebook were computed on free Google Colab, by the run that produced this copy. Run time and memory are in the technical section.

# Technical section

This section is not needed to read the results. It holds the state of the switch in this run, the input checksums and their outcome, the map from each item above to the study design's identifiers and to the files the run writes, the label texts, the full deciding tables with a graph of each result and its range, and the caveat on the boundary summaries. It ends with the run time and peak memory of every step of this run, and with the cell that collects the run's outputs in one archive. The study design's identifiers appear here.

## The switch in this run

In [ ]:
nb.switch_state()

## Input checksums

In [ ]:
nb.input_checksums()

## Map from each item to its design identifier

Each row names an item of the notebook, the identifier of the table or graph in the study's design that it shows, and the files this run writes for it, in its folders `tables/` and `figures/`. The two grids P2-F10 and P3-F10 draw the deciding results of P2-T3 and P3-T3; rows-split and rows-0035 are rows of the full sweep tables P3-T5 and P2-T5, read by their key columns.

| Order | Section | Item | Design identifier | Files the run writes |
|---|---|---|---|---|
| 1 | 3.1 | Naive review: the results that decide, at cost ratio 0.057, in one grid | P2-F10 (from P2-T3) | `figures/P2-F10_deciding_results.png`; `tables/P2-T3_deciding_results.csv`, `tables/P2-T3b_realistic_range_verdicts.csv`, `tables/P2-T3c_rule_outcomes.csv` |
| 2 | 3.2 | Analyst-aware deferral: the results that decide, at cost ratio 0.057, in one grid; both policies' overall false rejection rates beside the age-gap results against naive review | P3-F10 (from P3-T3) | `figures/P3-F10_deciding_results.png`; `tables/P3-T3_deciding_results.csv`, `tables/P3-T3b_realistic_range_verdicts.csv`, `tables/P3-T3c_rule_outcomes.csv` |
| 3 | 3.3 | Analyst-aware deferral minus naive review, expected loss, split into the alerts left to the classifier and the alerts reviewed | P3-F4 | `figures/P3-F4_split.png` |
| 4 | 3.3 | Teams B and C at capacity 0.75: the difference on the alerts naive review confirms without review | P3-T8 | `tables/P3-T8_point_figures.csv` |
| 5 | 3.3 | Analyst-aware deferral minus naive review, split into the alerts left to the classifier and the alerts reviewed, on both measures, with ranges and verdicts | rows of P3-T5 | `tables/rows-split.csv` |
| 6 | 4.1 | Analyst-aware deferral: how many alerts went to a reviewer, to which reviewer, and to the classifier | P3-T1 (routing) | `tables/P3-T1a_routing.csv` |
| 7 | 4.1 | How well the router tells apart the alerts each reviewer gets right from those the reviewer gets wrong | P3-T1 (ranking quality) | `tables/P3-T1b_ranking_quality_per_model_month6.csv`, `tables/P3-T1b_ranking_quality_per_model_month7.csv`, `tables/P3-T1c_ranking_quality_by_version.csv` |
| 8 | 5.1 | The teams: who is on them, and what teams B and C do in the test month | P2-T1 | `tables/P2-T1a_members_slot_order.csv`, `tables/P2-T1c_team_version_rates.csv`, `tables/P2-T1d_twins.csv`, `tables/P2-T1e_month7_items.csv` |
| 9 | 6.1 | Full automation and "confirm every alert": the three measures at each cost ratio | P1-T2 | `tables/P1-T2_baseline_by_cost_ratio.csv` |
| 10 | 6.1 | Full automation and "confirm every alert": the three measures against the cost ratio | P1-F1 | `figures/P1-F1_measures_vs_cost_ratio.png` |
| 11 | 6.2 | Naive review minus full automation, expected loss, against capacity | P2-F1 | `figures/P2-F1_expected_loss.png` |
| 12 | 6.2 | Analyst-aware deferral minus full automation, and minus naive review, expected loss, against capacity | P3-F1 | `figures/P3-F1_expected_loss.png` |
| 13 | 6.2 | Naive review minus full automation, expected loss, split into the reviewed alerts and the alerts beyond capacity | P2-F4 | `figures/P2-F4_split.png` |
| 14 | 6.3 | Naive review minus full automation, frauds accepted, against capacity | P2-F2 | `figures/P2-F2_accepted_frauds.png` |
| 15 | 6.3 | Analyst-aware deferral minus full automation, and minus naive review, frauds accepted, against capacity | P3-F2 | `figures/P3-F2_accepted_frauds.png` |
| 16 | 6.4 | Naive review minus full automation, size of the age gap, against capacity, with the overall false rejection rate | P2-F3 | `figures/P2-F3_absolute_age_gap.png` |
| 17 | 6.4 | Analyst-aware deferral minus full automation, and minus naive review, size of the age gap, against capacity, with the overall false rejection rate | P3-F3 | `figures/P3-F3_absolute_age_gap.png` |
| 18 | 6.5 | Naive review: where each team's difference from full automation crosses zero, and the verdict at each capacity | P2-T4 | `tables/P2-T4_capacity_boundaries.csv` |
| 19 | 6.5 | Analyst-aware deferral: where each team's difference from full automation and from naive review crosses zero, and the verdict at each capacity | P3-T4 | `tables/P3-T4_capacity_boundaries.csv` |
| 20 | 6.6 | Naive review: false rejection rates by age group for teams C and A against capacity, and how team C's bias changes between capacity 0.75 and full capacity | P2-F5 | `figures/P2-F5_bias_x_capacity.png`, `tables/P2-F5_bias_x_capacity_I.csv` |
| 21 | 6.6 | Analyst-aware deferral: how the difference between teams C and A changes between capacity 0.75 and full capacity | P3-T7 | `tables/P3-T7_bias_x_capacity.csv` |
| 22 | 6.7 | Share of alerts sent to a reviewer, by age group, analyst-aware deferral beside naive review | P3-T9 | `tables/P3-T9_access_by_age.csv` |
| 23 | 6.7 | Share of alerts sent to a reviewer against capacity, by age group, analyst-aware deferral and naive review | P3-F9 | `figures/P3-F9_access_by_age.png`, `tables/P3-F9_lines.csv` |
| 24 | 6.8 | Naive review: expected-loss verdict against full automation at each cost ratio and capacity; how each team's result holds across cost ratios | P2-F7 | `figures/P2-F7_cost_ratio_grid.png`, `tables/P2-F7_cost_ratio_labels.csv` |
| 25 | 6.8 | Analyst-aware deferral: expected-loss verdict against full automation at each cost ratio and capacity; how each team's result holds across cost ratios | P3-F6 | `figures/P3-F6_cost_ratio_grid.png`, `tables/P3-F6_cost_ratio_labels.csv` |
| 26 | 6.8 | Naive review and analyst-aware deferral against "confirm every alert", expected loss at cost ratio 0.035 | rows of P2-T5 and P3-T5 | `tables/rows-0035.csv` |
| 27 | 7 | Naive review: monitoring indicators against capacity, per team | P2-F6 | `figures/P2-F6_monitoring.png` |
| 28 | 7 | Analyst-aware deferral: monitoring indicators against capacity, per team, with criterion 1 as it turns out | P3-F5 | `figures/P3-F5_monitoring.png` |
| 29 | 8.3 | Naive review: every team, version, capacity and cost ratio, all measures | P2-T5 | `tables/P2-T5_full_sweep.csv` |
| 30 | 8.3 | Analyst-aware deferral: every team, version, capacity and cost ratio, all measures | P3-T5 | `tables/P3-T5_full_sweep.csv` |
| 31 | 8.4 | Naive review: observed "beats" and "does worse" against the number expected by chance | P2-F8 | `figures/P2-F8_chance_count.png` |
| 32 | 8.4 | Analyst-aware deferral: observed "beats" and "does worse" against the number expected by chance | P3-F7 | `figures/P3-F7_chance_count.png` |
| 33 | technical | Naive review: the results that decide, in full, with their widened ranges | P2-T3, P2-F9 | the three P2-T3 files of row 1; `figures/P2-F9_forest.png` |
| 34 | technical | Analyst-aware deferral: the results that decide, in full, with their widened ranges | P3-T3, P3-F8 | the three P3-T3 files of row 2; `figures/P3-F8_forest.png` |
| 35 | technical | Run time and memory of this run | T-T1 (this run's own records) | — (no file of its own; the table is formed from the run's `logs/steps.jsonl`, which the archive holds) |

## Label texts

The texts of the labels the tables carry, as the study's design sets them. Sentences that refer to the study's own checks and run records, and references to its internal decision numbers, are left out; nothing else is changed.

In [ ]:
nb.section("T-labels")

## The results that decide, in full

These are the deciding results of sections 3.1 and 3.2 as tables: for naive review the study design's table P2-T3, for analyst-aware deferral P3-T3, each from the files named in section 3. Every deciding result is listed with its widened range and its verdict. Below each table, a graph draws each result with its widened range: a square for the average of the five versions and dots for the five versions. The two results of naive review that follow from team C's rule are shown apart and are not findings. The verdicts are the same as in the grids of sections 3.1 and 3.2.

In [ ]:
nb.section("T-P2-T3")

In [ ]:
nb.section("T-P3-T3")

## Boundary summaries: caveat

The two tables of section 6.5 show a column headed "summary (can be misread; see the caveat in the technical section)" (`tables/P2-T4_capacity_boundaries.csv` and `tables/P3-T4_capacity_boundaries.csv`, column `interval summary`). The rule outcomes of naive review hold summaries of the same form in their table file, which is not displayed (`tables/P2-T3c_rule_outcomes.csv`, column `boundary_summary`). This run's verdict step forms these summaries once, and they are shown unchanged. Where a summary reads "does worse up to capacity X; never beats", with no "between:" list, it states only the highest capacity at which the comparison does worse, and that it never beats. It does not mean "does worse at every capacity up to X". The per-capacity verdicts govern: for the rule outcomes, those of `tables/P2-T3b_realistic_range_verdicts.csv`; for section 6.5, the capacity columns of its two tables (average of the five versions; the rows of each version are in the table files). No verdict, rule outcome or number changes.

In this run:
- `tables/P2-T3c_rule_outcomes.csv`: one row, the rule for reviewers who agree with the model on frauds accepted ("does worse up to capacity 1.00; never beats"): team B shows no difference at 0.65 to 0.95 and does worse only at 1.
- `tables/P2-T4_capacity_boundaries.csv`: no row has this form.
- `tables/P3-T4_capacity_boundaries.csv`: 27 rows have this form, all with X = 1.00. The 8 rows against naive review (team A version 2 on frauds accepted; team C version 2 on frauds accepted; team C versions 1 to 5 and the mean on the age gap) read "does worse" at every capacity. In the 19 rows against full automation, the only other verdict is "no difference shown": at capacity 0, where analyst-aware deferral is full automation, in every row; and also, on frauds accepted, for team A version 1 at 0.05, version 2 at 0.10, version 4 at 0.05 to 0.20 and version 5 at 0.05 to 0.15; for team B version 1 at 0.05, 0.10, 0.20 and 0.55, version 2 and the mean at 0.10; and for team C version 2 and the mean at 0.05 to 0.15. The other rows of this form (frauds accepted: team A version 3 and the mean, team B version 3, team C version 3; the age gap: team C versions 1 to 5 and the mean) differ from "does worse" at capacity 0 only.

## Run time and memory of this run

Each step's wall-clock time and peak memory, in bytes, computed from this run's own records while it ran. The set-up steps (environment, framework and FiFAR) run in the notebook's own process, whose memory is not measured step by step, so their memory cell shows a dash.

In [ ]:
nb.run_time_table()

## This run's outputs in one archive

The last cell collects a description of this run's outputs in one archive, `savehitl_returned_outputs.zip`, so that they can be kept after the session ends; on Colab it also offers the archive for download. Among other records of the run, it holds the table files, the per-column summaries of the run's case-level files (for columns of numbers: count, sum, sum of absolute values, minimum and maximum; for other columns: count, number of distinct values and a checksum of the values), the checksums of the router's 250 refitted models, and the run's time and memory records. It holds no case row of FiFAR. Nothing else in the notebook depends on this cell.

In [ ]:
nb.hand_back()